In [25]:
import pandas as pd
import numpy as np
import zipfile
from pathlib import Path
from statsmodels.tsa.arima.model import ARIMA
from sklearn.linear_model import LinearRegression
import warnings

warnings.filterwarnings("ignore")

# Extraccción

## Precipitación

### Consolidación Base Precipitación Casanare

In [26]:
import pandas as pd
import numpy as np
import zipfile
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

ruta = Path("Casanare_Precipitación.zip")

if ruta.suffix == ".zip":
    carpeta = ruta.with_suffix("")
    carpeta.mkdir(exist_ok=True)

    with zipfile.ZipFile(ruta, "r") as zip_ref:
        zip_ref.extractall(carpeta)
else:
    carpeta = ruta

archivos = list(carpeta.glob("*.xlsx"))

base = None

for archivo in archivos:
    print("Procesando:", archivo.name)

    nombre_estacion = pd.read_excel(
        archivo,
        header=None,
        nrows=2
    ).iloc[1, 1]

    if pd.isna(nombre_estacion):
        nombre_estacion = archivo.stem

    nombre_estacion = str(nombre_estacion)

    df = pd.read_excel(
        archivo,
        sheet_name="Datos",
        header=None,
        skiprows=7
    )

    df = df[[0, 2]]
    df.columns = ["fecha", nombre_estacion]

    df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")

    df[nombre_estacion] = (
        df[nombre_estacion]
        .astype(str)
        .str.replace(",", ".", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace("-", "", regex=False)
    )

    df[nombre_estacion] = pd.to_numeric(
        df[nombre_estacion],
        errors="coerce"
    )

    df = df.dropna(subset=["fecha"])

    if base is None:
        base = df
    else:
        base = pd.merge(
            base,
            df,
            on="fecha",
            how="outer"
        )

base = base.sort_values("fecha").reset_index(drop=True)

#Dato solo desde 2010
base = base[
    base["fecha"] >= pd.Timestamp("2010-01-01")
].reset_index(drop=True)

original = base.copy()

print("Base consolidada")
print(
    "Fechas:",
    base["fecha"].min(),
    "hasta",
    base["fecha"].max()
)

original = base.copy()

print("Base consolidada")

Procesando: 35095110.xlsx
Procesando: 35195030.xlsx
Procesando: 35215020.xlsx
Procesando: 35225030.xlsx
Procesando: 35235040.xlsx
Procesando: 35235050.xlsx
Procesando: 36015010.xlsx
Base consolidada
Fechas: 2010-01-01 00:00:00 hasta 2025-12-31 00:00:00
Base consolidada


### Prueba de estacionariedad y estacionalidad

In [27]:
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose

resumen_pruebas = []

for columna in base.columns:

    if columna == "fecha":
        continue

    serie = base[["fecha", columna]].dropna()
    serie = serie.set_index("fecha")[columna]

    if len(serie) < 24:
        resumen_pruebas.append({
            "estacion": columna,
            "n_observaciones": len(serie),
            "ADF_p_value": np.nan,
            "es_estacionaria": "No evaluable",
            "observacion": "Muy pocos datos"
        })
        continue

    resultado_adf = adfuller(serie)

    p_value = resultado_adf[1]

    es_estacionaria = "Sí" if p_value < 0.05 else "No"

    resumen_pruebas.append({
        "estacion": columna,
        "n_observaciones": len(serie),
        "ADF_p_value": p_value,
        "es_estacionaria": es_estacionaria,
        "observacion": "Evaluada"
    })

resumen_pruebas = pd.DataFrame(resumen_pruebas)

resumen_pruebas

,estacion,n_observaciones,ADF_p_value,es_estacionaria,observacion
0,HUERTA LA GRANDE [35095110],5835,2.301163e-12,Sí,Evaluada
1,AGUAZUL [35195030],4382,8.399109e-12,Sí,Evaluada
2,AEROPUERTO YOPAL - AUT [35215020],5671,3.908391e-15,Sí,Evaluada
3,MODULOS - AUT [35225030],5650,1.108505e-12,Sí,Evaluada
4,TRINIDAD - AUT [35235040],5688,3.740628e-17,Sí,Evaluada
5,TAMARA - AUT [35235050],5816,5.729285e-10,Sí,Evaluada
6,PAZ DE ARIPORO [36015010],5483,1.650177e-13,Sí,Evaluada


### ARIMA para vacios


In [28]:
from statsmodels.tsa.arima.model import ARIMA
!pip install pmdarima
from pmdarima import auto_arima

def imputar_arima(df, columna):

    datos = df[["fecha", columna]].copy()
    datos = datos.sort_values("fecha")
    datos = datos.set_index("fecha")

    y = pd.to_numeric(
        datos[columna],
        errors="coerce"
    )

    # Guardar ubicación de vacíos originales
    mask_na_original = y.isna()

    # Si hay pocos datos usar interpolación
    if y.notna().sum() < 30:

        y_final = (
            y.interpolate("linear")
            .ffill()
            .bfill()
        )

        return (
            y_final.reset_index(drop=True),
            "Interpolación (pocos datos)"
        )

    # Interpolación temporal preliminar
    # necesaria para entrenar ARIMA
    y_temp = (
        y.interpolate("linear")
        .ffill()
        .bfill()
    )

    try:

        # Selección automática de p,d,q
        modelo_auto = auto_arima(
            y_temp,
            seasonal=False,
            d=0,   # ADF mostró estacionariedad
            trace=False,
            suppress_warnings=True
        )

        p,d,q = modelo_auto.order

        modelo = ARIMA(
            y_temp,
            order=(p,d,q)
        )

        ajuste = modelo.fit()

        predicciones = ajuste.predict(
            start=0,
            end=len(y_temp)-1
        )

        y_final = y.copy()

        # Solo reemplazar vacíos originales
        y_final.loc[
            mask_na_original
        ] = predicciones.loc[
            mask_na_original
        ]

        return (
            y_final.reset_index(drop=True),
            f"ARIMA{(p,d,q)}"
        )

    except Exception as e:

        y_final = (
            y.interpolate("linear")
            .ffill()
            .bfill()
        )

        return (
            y_final.reset_index(drop=True),
            f"Interpolación por error: {e}"
        )

In [29]:
def seleccionar_mejor_arima(serie):

    mejor_aic = np.inf
    mejor_orden = None

    ordenes = [
        (1, 0, 0),
        (0, 0, 1),
        (1, 0, 1),
        (2, 0, 0),
        (0, 0, 2),
        (2, 0, 1),
        (1, 0, 2)
    ]

    for orden in ordenes:

        try:
            modelo = ARIMA(
                serie,
                order=orden
            )

            resultado = modelo.fit(
                method_kwargs={"maxiter": 50}
            )

            if resultado.aic < mejor_aic:
                mejor_aic = resultado.aic
                mejor_orden = orden

        except:
            continue

    if mejor_orden is None:
        mejor_orden = (1, 0, 0)

    return mejor_orden

In [30]:
def imputar_arima(df, columna):

    datos = df[["fecha", columna]].copy()
    datos = datos.sort_values("fecha")

    datos = datos.set_index("fecha")
    datos.index = pd.to_datetime(datos.index)

    y = pd.to_numeric(
        datos[columna],
        errors="coerce"
    )

    mask_na_original = y.isna()

    # Si hay pocos datos, usar interpolación temporal
    if y.notna().sum() < 30:

        y_final = (
            y
            .interpolate(method="time")
            .ffill()
            .bfill()
        )

        return (
            y_final.reset_index(drop=True),
            "Interpolación temporal por pocos datos"
        )

    # Interpolación preliminar para que ARIMA pueda entrenarse
    y_temp = (
        y
        .interpolate(method="time")
        .ffill()
        .bfill()
    )

    try:
        orden = seleccionar_mejor_arima(y_temp)

        modelo = ARIMA(
            y_temp,
            order=orden
        )

        ajuste = modelo.fit(
            method_kwargs={"maxiter": 50}
        )

        predicciones = ajuste.predict(
            start=0,
            end=len(y_temp) - 1
        )

        y_final = y.copy()

        # Solo reemplaza los datos que originalmente estaban vacíos
        y_final.loc[mask_na_original] = predicciones.loc[mask_na_original]

        # Respaldo final: si queda algún vacío, se completa
        y_final = (
            y_final
            .interpolate(method="time")
            .ffill()
            .bfill()
        )

        return (
            y_final.reset_index(drop=True),
            f"ARIMA{orden} + respaldo temporal"
        )

    except Exception as e:

        y_final = (
            y
            .interpolate(method="time")
            .ffill()
            .bfill()
        )

        return (
            y_final.reset_index(drop=True),
            f"Interpolación temporal porque ARIMA falló: {e}"
        )

In [31]:
completado = base.copy()

marca_imputacion = pd.DataFrame()
marca_imputacion["fecha"] = base["fecha"]

resumen_modelos = []

for columna in base.columns:

    if columna == "fecha":
        continue

    print("Imputando:", columna)

    marca_imputacion[columna] = base[columna].isna()

    serie_completa, metodo = imputar_arima(
        base,
        columna
    )

    completado[columna] = serie_completa

    resumen_modelos.append({
        "estacion": columna,
        "modelo_usado": metodo,
        "observaciones_originales": base[columna].notna().sum(),
        "valores_imputados": base[columna].isna().sum(),
        "porcentaje_imputado": round(
            100 * base[columna].isna().sum() / len(base),
            2
        ),
        "faltantes_finales": completado[columna].isna().sum()
    })

resumen_modelos = pd.DataFrame(resumen_modelos)

print("Imputación terminada")

Imputando: HUERTA LA GRANDE [35095110]
Imputando: AGUAZUL [35195030]
Imputando: AEROPUERTO YOPAL - AUT [35215020]
Imputando: MODULOS - AUT [35225030]
Imputando: TRINIDAD - AUT [35235040]
Imputando: TAMARA - AUT [35235050]
Imputando: PAZ DE ARIPORO [36015010]
Imputación terminada


In [32]:
faltantes_finales = completado.isna().sum()

faltantes_finales

fecha                                0
HUERTA LA GRANDE [35095110]          0
AGUAZUL [35195030]                   0
AEROPUERTO YOPAL - AUT [35215020]    0
MODULOS - AUT [35225030]             0
TRINIDAD - AUT [35235040]            0
TAMARA - AUT [35235050]              0
PAZ DE ARIPORO [36015010]            0
dtype: int64

### Exportar Excel

In [33]:
with pd.ExcelWriter(
    "Casanare_Precipitación.xlsx",
    engine="openpyxl"
) as writer:

    completado.to_excel(
        writer,
        sheet_name="Completado",
        index=False
    )

    original.to_excel(
        writer,
        sheet_name="Original",
        index=False
    )

    marca_imputacion.to_excel(
        writer,
        sheet_name="Marca_imputacion",
        index=False
    )

    resumen_pruebas.to_excel(
        writer,
        sheet_name="Pruebas_ADF",
        index=False
    )

    resumen_modelos.to_excel(
        writer,
        sheet_name="Resumen_modelos",
        index=False
    )

print("Archivo creado: Casanare_Precipitación.xlsx")

Archivo creado: Casanare_Precipitación.xlsx


## Temperatura Máxima

### Consolidación Base Precipitación Casanare

In [34]:
import pandas as pd
import numpy as np
import zipfile
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

ruta = Path("Casanare_Precipitación.zip")

if ruta.suffix == ".zip":
    carpeta = ruta.with_suffix("")
    carpeta.mkdir(exist_ok=True)

    with zipfile.ZipFile(ruta, "r") as zip_ref:
        zip_ref.extractall(carpeta)
else:
    carpeta = ruta

archivos = list(carpeta.glob("*.xlsx"))

base = None

for archivo in archivos:
    print("Procesando:", archivo.name)

    nombre_estacion = pd.read_excel(
        archivo,
        header=None,
        nrows=2
    ).iloc[1, 1]

    if pd.isna(nombre_estacion):
        nombre_estacion = archivo.stem

    nombre_estacion = str(nombre_estacion)

    df = pd.read_excel(
        archivo,
        sheet_name="Datos",
        header=None,
        skiprows=7
    )

    df = df[[0, 2]]
    df.columns = ["fecha", nombre_estacion]

    df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")

    df[nombre_estacion] = (
        df[nombre_estacion]
        .astype(str)
        .str.replace(",", ".", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace("-", "", regex=False)
    )

    df[nombre_estacion] = pd.to_numeric(
        df[nombre_estacion],
        errors="coerce"
    )

    df = df.dropna(subset=["fecha"])

    if base is None:
        base = df
    else:
        base = pd.merge(
            base,
            df,
            on="fecha",
            how="outer"
        )

base = base.sort_values("fecha").reset_index(drop=True)

#Dato solo desde 2010
base = base[
    base["fecha"] >= pd.Timestamp("2010-01-01")
].reset_index(drop=True)

original = base.copy()

print("Base consolidada")
print(
    "Fechas:",
    base["fecha"].min(),
    "hasta",
    base["fecha"].max()
)

original = base.copy()

print("Base consolidada")

Procesando: 35095110.xlsx
Procesando: 35195030.xlsx
Procesando: 35215020.xlsx
Procesando: 35225030.xlsx
Procesando: 35235040.xlsx
Procesando: 35235050.xlsx
Procesando: 36015010.xlsx
Base consolidada
Fechas: 2010-01-01 00:00:00 hasta 2025-12-31 00:00:00
Base consolidada


### Prueba de estacionariedad y estacionalidad

In [35]:
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose

resumen_pruebas = []

for columna in base.columns:

    if columna == "fecha":
        continue

    serie = base[["fecha", columna]].dropna()
    serie = serie.set_index("fecha")[columna]

    if len(serie) < 24:
        resumen_pruebas.append({
            "estacion": columna,
            "n_observaciones": len(serie),
            "ADF_p_value": np.nan,
            "es_estacionaria": "No evaluable",
            "observacion": "Muy pocos datos"
        })
        continue

    resultado_adf = adfuller(serie)

    p_value = resultado_adf[1]

    es_estacionaria = "Sí" if p_value < 0.05 else "No"

    resumen_pruebas.append({
        "estacion": columna,
        "n_observaciones": len(serie),
        "ADF_p_value": p_value,
        "es_estacionaria": es_estacionaria,
        "observacion": "Evaluada"
    })

resumen_pruebas = pd.DataFrame(resumen_pruebas)

resumen_pruebas

,estacion,n_observaciones,ADF_p_value,es_estacionaria,observacion
0,HUERTA LA GRANDE [35095110],5835,2.301163e-12,Sí,Evaluada
1,AGUAZUL [35195030],4382,8.399109e-12,Sí,Evaluada
2,AEROPUERTO YOPAL - AUT [35215020],5671,3.908391e-15,Sí,Evaluada
3,MODULOS - AUT [35225030],5650,1.108505e-12,Sí,Evaluada
4,TRINIDAD - AUT [35235040],5688,3.740628e-17,Sí,Evaluada
5,TAMARA - AUT [35235050],5816,5.729285e-10,Sí,Evaluada
6,PAZ DE ARIPORO [36015010],5483,1.650177e-13,Sí,Evaluada


In [36]:
completado = base.copy()

marca_imputacion = pd.DataFrame()
marca_imputacion["fecha"] = base["fecha"]

resumen_modelos = []

for columna in base.columns:

    if columna == "fecha":
        continue

    print("Imputando:", columna)

    marca_imputacion[columna] = base[columna].isna()

    serie_completa, metodo = imputar_arima(
        base,
        columna
    )

    completado[columna] = serie_completa

    resumen_modelos.append({
        "estacion": columna,
        "modelo_usado": metodo,
        "observaciones_originales": base[columna].notna().sum(),
        "valores_imputados": base[columna].isna().sum(),
        "porcentaje_imputado": round(
            100 * base[columna].isna().sum() / len(base),
            2
        ),
        "faltantes_finales": completado[columna].isna().sum()
    })

resumen_modelos = pd.DataFrame(resumen_modelos)

print("Imputación terminada")

Imputando: HUERTA LA GRANDE [35095110]
Imputando: AGUAZUL [35195030]
Imputando: AEROPUERTO YOPAL - AUT [35215020]
Imputando: MODULOS - AUT [35225030]
Imputando: TRINIDAD - AUT [35235040]
Imputando: TAMARA - AUT [35235050]
Imputando: PAZ DE ARIPORO [36015010]
Imputación terminada


In [37]:
faltantes_finales = completado.isna().sum()

faltantes_finales

fecha                                0
HUERTA LA GRANDE [35095110]          0
AGUAZUL [35195030]                   0
AEROPUERTO YOPAL - AUT [35215020]    0
MODULOS - AUT [35225030]             0
TRINIDAD - AUT [35235040]            0
TAMARA - AUT [35235050]              0
PAZ DE ARIPORO [36015010]            0
dtype: int64

### Exportar Excel

In [38]:
with pd.ExcelWriter(
    "Casanare_TempMax.xlsx",
    engine="openpyxl"
) as writer:

    completado.to_excel(
        writer,
        sheet_name="Completado",
        index=False
    )

    original.to_excel(
        writer,
        sheet_name="Original",
        index=False
    )

    marca_imputacion.to_excel(
        writer,
        sheet_name="Marca_imputacion",
        index=False
    )

    resumen_pruebas.to_excel(
        writer,
        sheet_name="Pruebas_ADF",
        index=False
    )

    resumen_modelos.to_excel(
        writer,
        sheet_name="Resumen_modelos",
        index=False
    )

print("Archivo creado: Casanare_TempMax.xlsx")

Archivo creado: Casanare_TempMax.xlsx


## Temperatura Mínima

### Consolidación Base Precipitación Casanare

In [39]:
import pandas as pd
import numpy as np
import zipfile
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

ruta = Path("Casanare_TempMin.zip")

if ruta.suffix == ".zip":
    carpeta = ruta.with_suffix("")
    carpeta.mkdir(exist_ok=True)

    with zipfile.ZipFile(ruta, "r") as zip_ref:
        zip_ref.extractall(carpeta)
else:
    carpeta = ruta

archivos = list(carpeta.glob("*.xlsx"))

base = None

for archivo in archivos:
    print("Procesando:", archivo.name)

    nombre_estacion = pd.read_excel(
        archivo,
        header=None,
        nrows=2
    ).iloc[1, 1]

    if pd.isna(nombre_estacion):
        nombre_estacion = archivo.stem

    nombre_estacion = str(nombre_estacion)

    df = pd.read_excel(
        archivo,
        sheet_name="Datos",
        header=None,
        skiprows=7
    )

    df = df[[0, 2]]
    df.columns = ["fecha", nombre_estacion]

    df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")

    df[nombre_estacion] = (
        df[nombre_estacion]
        .astype(str)
        .str.replace(",", ".", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace("-", "", regex=False)
    )

    df[nombre_estacion] = pd.to_numeric(
        df[nombre_estacion],
        errors="coerce"
    )

    df = df.dropna(subset=["fecha"])

    if base is None:
        base = df
    else:
        base = pd.merge(
            base,
            df,
            on="fecha",
            how="outer"
        )

base = base.sort_values("fecha").reset_index(drop=True)

#Dato solo desde 2010
base = base[
    base["fecha"] >= pd.Timestamp("2010-01-01")
].reset_index(drop=True)

original = base.copy()

print("Base consolidada")
print(
    "Fechas:",
    base["fecha"].min(),
    "hasta",
    base["fecha"].max()
)

original = base.copy()

print("Base consolidada")

Procesando: 35095110.xlsx
Procesando: 35195030.xlsx
Procesando: 35215020.xlsx
Procesando: 35225030.xlsx
Procesando: 35235040.xlsx
Procesando: 35235050.xlsx
Procesando: 36015010.xlsx
Base consolidada
Fechas: 2010-01-01 00:00:00 hasta 2025-12-31 00:00:00
Base consolidada


### Prueba de estacionariedad y estacionalidad

In [40]:
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose

resumen_pruebas = []

for columna in base.columns:

    if columna == "fecha":
        continue

    serie = base[["fecha", columna]].dropna()
    serie = serie.set_index("fecha")[columna]

    if len(serie) < 24:
        resumen_pruebas.append({
            "estacion": columna,
            "n_observaciones": len(serie),
            "ADF_p_value": np.nan,
            "es_estacionaria": "No evaluable",
            "observacion": "Muy pocos datos"
        })
        continue

    resultado_adf = adfuller(serie)

    p_value = resultado_adf[1]

    es_estacionaria = "Sí" if p_value < 0.05 else "No"

    resumen_pruebas.append({
        "estacion": columna,
        "n_observaciones": len(serie),
        "ADF_p_value": p_value,
        "es_estacionaria": es_estacionaria,
        "observacion": "Evaluada"
    })

resumen_pruebas = pd.DataFrame(resumen_pruebas)

resumen_pruebas

,estacion,n_observaciones,ADF_p_value,es_estacionaria,observacion
0,HUERTA LA GRANDE [35095110],5467,1.512952e-12,Sí,Evaluada
1,AGUAZUL [35195030],3881,3.101684e-07,Sí,Evaluada
2,AEROPUERTO YOPAL - AUT [35215020],5211,4.015986e-05,Sí,Evaluada
3,MODULOS - AUT [35225030],5449,1.320858e-11,Sí,Evaluada
4,TRINIDAD - AUT [35235040],5490,1.181274e-11,Sí,Evaluada
5,TAMARA - AUT [35235050],4962,8.135507e-06,Sí,Evaluada
6,PAZ DE ARIPORO [36015010],5283,3.651233e-13,Sí,Evaluada


In [41]:
completado = base.copy()

marca_imputacion = pd.DataFrame()
marca_imputacion["fecha"] = base["fecha"]

resumen_modelos = []

for columna in base.columns:

    if columna == "fecha":
        continue

    print("Imputando:", columna)

    marca_imputacion[columna] = base[columna].isna()

    serie_completa, metodo = imputar_arima(
        base,
        columna
    )

    completado[columna] = serie_completa

    resumen_modelos.append({
        "estacion": columna,
        "modelo_usado": metodo,
        "observaciones_originales": base[columna].notna().sum(),
        "valores_imputados": base[columna].isna().sum(),
        "porcentaje_imputado": round(
            100 * base[columna].isna().sum() / len(base),
            2
        ),
        "faltantes_finales": completado[columna].isna().sum()
    })

resumen_modelos = pd.DataFrame(resumen_modelos)

print("Imputación terminada")

Imputando: HUERTA LA GRANDE [35095110]
Imputando: AGUAZUL [35195030]
Imputando: AEROPUERTO YOPAL - AUT [35215020]
Imputando: MODULOS - AUT [35225030]
Imputando: TRINIDAD - AUT [35235040]
Imputando: TAMARA - AUT [35235050]
Imputando: PAZ DE ARIPORO [36015010]
Imputación terminada


In [42]:
faltantes_finales = completado.isna().sum()

faltantes_finales

fecha                                0
HUERTA LA GRANDE [35095110]          0
AGUAZUL [35195030]                   0
AEROPUERTO YOPAL - AUT [35215020]    0
MODULOS - AUT [35225030]             0
TRINIDAD - AUT [35235040]            0
TAMARA - AUT [35235050]              0
PAZ DE ARIPORO [36015010]            0
dtype: int64

### Exportar Excel

In [43]:
with pd.ExcelWriter(
    "Casanare_TempMin.xlsx",
    engine="openpyxl"
) as writer:

    completado.to_excel(
        writer,
        sheet_name="Completado",
        index=False
    )

    original.to_excel(
        writer,
        sheet_name="Original",
        index=False
    )

    marca_imputacion.to_excel(
        writer,
        sheet_name="Marca_imputacion",
        index=False
    )

    resumen_pruebas.to_excel(
        writer,
        sheet_name="Pruebas_ADF",
        index=False
    )

    resumen_modelos.to_excel(
        writer,
        sheet_name="Resumen_modelos",
        index=False
    )

print("Archivo creado: Casanare_TempMin.xlsx")

Archivo creado: Casanare_TempMin.xlsx


# Cosolidaación Capa A

## Promediar estaciones

In [44]:
archivos = {
    "Precipitacion": "Casanare_Precipitación.xlsx",
    "TempMax": "Casanare_TempMax.xlsx",
    "TempMin": "Casanare_TempMin.xlsx"
}

In [45]:
#CALCULAR PROMEDIOS
def calcular_promedio_diario(df, nombre_variable):

    df = df.copy()

    # Normalizar nombre de fecha
    if "fecha" in df.columns:
        col_fecha = "fecha"
    elif "date" in df.columns:
        col_fecha = "date"
    else:
        col_fecha = df.columns[0]

    df[col_fecha] = pd.to_datetime(
        df[col_fecha],
        errors="coerce"
    )

    columnas_valores = [
        col for col in df.columns
        if col != col_fecha
    ]

    for col in columnas_valores:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

    promedio = pd.DataFrame()
    promedio["date"] = df[col_fecha]
    promedio[nombre_variable] = df[columnas_valores].mean(axis=1)

    return promedio

In [46]:
#Leer hojas de completado
hojas_completado = {}
promedios = []

for nombre, archivo in archivos.items():

    df = pd.read_excel(
        archivo,
        sheet_name="Completado"
    )

    hojas_completado[nombre] = df

    promedio = calcular_promedio_diario(
        df,
        nombre
    )

    promedios.append(promedio)

In [47]:
#Unir promedioas
promedios_final = promedios[0]

for promedio in promedios[1:]:
    promedios_final = promedios_final.merge(
        promedio,
        on="date",
        how="outer"
    )

promedios_final = promedios_final.sort_values("date")

promedios_final = promedios_final.rename(columns={
    "Precipitacion": "precip",
    "TempMax": "t_max",
    "TempMin": "t_min"
})

promedios_final["department"] = "Casanare"
promedios_final["zone"] = "Llanos"

promedios_final["t_mean"] = (
    promedios_final["t_min"] + promedios_final["t_max"]
) / 2

promedios_final["source"] = "ideam"

promedios_final = promedios_final[
    [
        "department",
        "zone",
        "date",
        "t_min",
        "t_max",
        "t_mean",
        "precip",
        "source"
    ]
]

In [48]:
#Guardar nuevo escel
with pd.ExcelWriter(
    "Casanare_Completo.xlsx",
    engine="openpyxl"
) as writer:

    for nombre, df in hojas_completado.items():
        df.to_excel(
            writer,
            sheet_name=nombre,
            index=False
        )

    promedios_final.to_excel(
        writer,
        sheet_name="Promedios_diarios",
        index=False
    )

print("Archivo creado: Casanare_Completo.xlsx")

Archivo creado: Casanare_Completo.xlsx
